# RT-DETRv2-X Pill Detection Competition Notebook (Colab CUDA)

이 노트북은 Google Colab CUDA 런타임에서 `mAP@[0.75:0.95]` 평가 대회용 **RT-DETRv2-X detector 학습 + 추론 + submission.csv 생성**을 실행하도록 분리한 버전입니다.

핵심 흐름:

```text
kagglehub.competition_download('ai12-level1-project')
  ↓
sprint_ai_project1_data/ 자동 탐색 또는 압축 해제
  ↓
내장 corrections.json 생성 + train.csv 보정 적용
  ↓
COCO detection format 변환
  ↓
RT-DETRv2-X fine-tuning on CUDA
  ↓
validation mAP@[0.75:0.95] 확인
  ↓
test inference
  ↓
submission.csv
```

> Colab에서는 `런타임 > 런타임 유형 변경 > GPU`를 먼저 켜세요. T4 기준 안정 기본값은 batch 2이고, L4/A100이면 `TOTAL_BATCH_SIZE`를 4 이상으로 올려볼 수 있습니다.


## 0. 사용 전 체크

아래만 먼저 맞추면 됩니다.

1. Colab 런타임을 GPU로 설정
2. Kaggle 인증 준비: Colab Secrets에 `KAGGLE_USERNAME`, `KAGGLE_KEY`를 넣거나 `/content/kaggle.json` 업로드
3. 첫 데이터 셀이 `kagglehub.competition_download('ai12-level1-project')`로 대회 파일을 받음

`corrections.json`은 노트북 안에 내장되어 있어서 파일을 따로 올리지 않아도 자동 생성됩니다. 이미 같은 경로에 `corrections.json`이 있으면 그 파일을 우선 사용합니다.


In [ ]:
# =========================
# Notebook Parameters - Colab CUDA
# =========================
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DATA_DIR_NAME = "sprint_ai_project1_data"
DATA_ZIP_NAME = "ai12-level1-project.zip"
KAGGLE_COMPETITION = "ai12-level1-project"
FORCE_KAGGLE_DOWNLOAD = False

# Colab은 기본 cwd가 /content입니다. Drive를 mount했다면 후보 경로에서 자동 탐색합니다.
def has_project_payload(path: Path):
    return (
        (path / DATA_DIR_NAME).exists()
        or (path / DATA_ZIP_NAME).exists()
        or (path / "corrections.json").exists()
    )

def find_project_dir(start: Path | None = None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    if IN_COLAB:
        candidates += [
            Path("/content"),
            Path("/content/detectionproject"),
            Path("/content/drive/MyDrive/detectionproject"),
            Path("/content/drive/MyDrive/AIENGINEERCOURSE/detectionproject"),
        ]
    for candidate in candidates:
        if candidate.exists() and has_project_payload(candidate):
            return candidate
    return Path("/content") if IN_COLAB else start

PROJECT_DIR = find_project_dir()
os.chdir(PROJECT_DIR)

COMPETITION_INPUT_DIR = PROJECT_DIR / DATA_DIR_NAME
CORRECTIONS_PATH = PROJECT_DIR / "corrections.json"

TRAIN_CSV_NAME = "train.csv"
TRAIN_IMAGE_DIR_NAME = "train_images"
TEST_IMAGE_DIR_NAME = "test_images"
TRAIN_ANNOTATION_DIR_NAME = "train_annotations"

# RT-DETRv2 model size: "x" = RT-DETRv2-X, "l" = RT-DETRv2-L fallback
MODEL_SIZE = "x"

# 고 IoU 대회라 640 baseline 후 800/960도 실험 추천
IMAGE_SIZE = 640

# 대회 조건상 이미지당 최대 객체 수가 4개라면 4
MAX_DET_PER_IMAGE = 4

# mAP는 ranking이 중요하므로 threshold는 낮게 두고 top-k로 제한
SCORE_THR = 0.001

VAL_SIZE = 0.15
RANDOM_SEED = 42

EPOCHS = 50

# T4 안정 기본값. L4/A100이면 4 이상으로 올려도 됩니다.
TOTAL_BATCH_SIZE = 2
NUM_WORKERS = 2

# Colab CUDA 기본값
PROCESSOR = "cuda"
USE_AMP = True

# Colab Drive는 느릴 수 있어 학습 산출물은 /content/working에 둡니다.
WORK_DIR = Path("/content/working") if IN_COLAB else PROJECT_DIR / "working"
DATA_WORK_DIR = WORK_DIR / "pill_coco"
REPO_DIR = WORK_DIR / "RT-DETR"
RTDETRV2_DIR = REPO_DIR / "rtdetrv2_pytorch"

print("IN_COLAB:", IN_COLAB)
print("PROJECT_DIR:", PROJECT_DIR)
print("COMPETITION_INPUT_DIR:", COMPETITION_INPUT_DIR)
print("CORRECTIONS_PATH:", CORRECTIONS_PATH, CORRECTIONS_PATH.exists())
print("WORK_DIR:", WORK_DIR)
print("PROCESSOR:", PROCESSOR)
print("USE_AMP:", USE_AMP)
print("TOTAL_BATCH_SIZE:", TOTAL_BATCH_SIZE)
print("NUM_WORKERS:", NUM_WORKERS)
print("KAGGLE_COMPETITION:", KAGGLE_COMPETITION)
print("FORCE_KAGGLE_DOWNLOAD:", FORCE_KAGGLE_DOWNLOAD)


## 0.1 Embedded Annotation Corrections

Colab에 `corrections.json`을 따로 올리지 않아도 같은 보정 내용을 재현하도록 노트북에 내장합니다.


In [ ]:
import json

EMBEDDED_CORRECTIONS_JSON = r"""
{
  "bbox_corrections": [
    {
      "ann_id": 4683,
      "image_id": 1228,
      "category_id": 16548,
      "file_name": "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png",
      "original": [
        88,
        255,
        366,
        209
      ],
      "corrected": [
        94,
        861,
        241,
        238
      ],
      "reason": "좌표 오류"
    },
    {
      "ann_id": 5178,
      "image_id": 1360,
      "category_id": 19232,
      "file_name": "K-003351-019232-029667_0_2_1_2_70_000_200.png",
      "original": [
        523,
        201,
        232,
        241
      ],
      "corrected": [
        524,
        202,
        231,
        334
      ],
      "reason": "좌표 오류"
    },
    {
      "ann_id": 5662,
      "image_id": 1491,
      "category_id": 3351,
      "file_name": "K-003351-020014-020238_0_2_0_2_90_000_200.png",
      "original": [
        185,
        249,
        188,
        183
      ],
      "corrected": [
        387,
        249,
        182,
        183
      ],
      "reason": "좌표 오류"
    },
    {
      "ann_id": 3444,
      "image_id": 907,
      "category_id": 3351,
      "file_name": "K-003351-020238-031863_0_2_0_2_70_000_200.png",
      "original": [
        446,
        844,
        190,
        191
      ],
      "corrected": [
        579,
        291,
        229,
        215
      ],
      "reason": "좌표 오류"
    },
    {
      "ann_id": 791,
      "image_id": 208,
      "category_id": 3351,
      "file_name": "K-003351-029667-031863_0_2_0_2_70_000_200.png",
      "original": [
        178,
        249,
        205,
        203
      ],
      "corrected": [
        370,
        855,
        190,
        191
      ],
      "reason": "좌표 오류"
    },
    {
      "ann_id": 903,
      "image_id": 239,
      "category_id": 18357,
      "file_name": "K-003351-016262-018357_0_2_0_2_75_000_200.png",
      "original": [
        6567,
        625,
        311,
        315
      ],
      "corrected": [
        567,
        625,
        311,
        315
      ],
      "reason": "좌표 오류 (x좌표 6567 → 567)"
    }
  ],
  "bbox_additions": [
    {
      "ann_id": 5692,
      "image_id": 193,
      "category_id": 3351,
      "bbox": [
        381,
        833,
        196,
        191
      ],
      "area": 37436,
      "file_name": "K-003351-013900-021325_0_2_0_2_70_000_200.png",
      "reason": "bbox 누락"
    },
    {
      "ann_id": 5693,
      "image_id": 1267,
      "category_id": 3351,
      "bbox": [
        423,
        868,
        200,
        188
      ],
      "area": 37600,
      "file_name": "K-003351-013900-036637_0_2_0_2_70_000_200.png",
      "reason": "bbox 누락"
    },
    {
      "ann_id": 5694,
      "image_id": 783,
      "category_id": 20014,
      "bbox": [
        53,
        714,
        335,
        343
      ],
      "area": 114905,
      "file_name": "K-003351-020014-022074_0_2_0_2_90_000_200.png",
      "reason": "bbox 누락"
    },
    {
      "ann_id": 5695,
      "image_id": 1383,
      "category_id": 32310,
      "bbox": [
        569,
        809,
        384,
        259
      ],
      "area": 99456,
      "file_name": "K-003351-021325-032310_0_2_0_2_90_000_200.png",
      "reason": "bbox 누락"
    },
    {
      "ann_id": 5696,
      "image_id": 1432,
      "category_id": 3351,
      "bbox": [
        377,
        857,
        193,
        192
      ],
      "area": 37056,
      "file_name": "K-003351-032310-038162_0_2_0_2_70_000_200.png",
      "reason": "bbox 누락"
    },
    {
      "ann_id": 5697,
      "image_id": 1405,
      "category_id": 33880,
      "bbox": [
        55,
        589,
        319,
        433
      ],
      "area": 138127,
      "file_name": "K-003351-033880-038162_0_2_0_2_75_000_200.png",
      "reason": "bbox 누락"
    },
    {
      "ann_id": 5698,
      "image_id": 16,
      "category_id": 3351,
      "bbox": [
        452,
        868,
        195,
        195
      ],
      "area": 38025,
      "file_name": "K-003351-035206-041768_0_2_0_2_70_000_200.png",
      "reason": "bbox 누락"
    },
    {
      "ann_id": 5699,
      "image_id": 1258,
      "category_id": 4543,
      "bbox": [
        625,
        191,
        216,
        210
      ],
      "area": 45360,
      "file_name": "K-003544-004543-012247-016548_0_2_0_2_90_000_200.png",
      "reason": "bbox 누락"
    }
  ],
  "category_corrections": [
    {
      "ann_id": 712,
      "image_id": 377,
      "file_name": "K-003483-025367-027733-035206_0_2_0_2_75_000_200.png",
      "original_category_id": 27733,
      "corrected_category_id": 35206,
      "bbox": [
        565,
        119,
        257,
        380
      ],
      "reason": "category swap: imprint 337 is Atozet, not Twynsta"
    },
    {
      "ann_id": 715,
      "image_id": 377,
      "file_name": "K-003483-025367-027733-035206_0_2_0_2_75_000_200.png",
      "original_category_id": 35206,
      "corrected_category_id": 27733,
      "bbox": [
        87,
        726,
        321,
        283
      ],
      "reason": "category swap: imprint A1 is Twynsta, not Atozet"
    }
  ]
}
""".strip()

def ensure_corrections_json(path: Path):
    embedded = json.loads(EMBEDDED_CORRECTIONS_JSON)
    if path.exists():
        existing = json.loads(path.read_text(encoding="utf-8"))
        changed = False
        for key in ["bbox_corrections", "bbox_additions", "category_corrections"]:
            existing_items = existing.setdefault(key, [])
            existing_ann_ids = {int(item.get("ann_id")) for item in existing_items if "ann_id" in item}
            for item in embedded.get(key, []):
                ann_id = int(item.get("ann_id"))
                if ann_id not in existing_ann_ids:
                    existing_items.append(item)
                    existing_ann_ids.add(ann_id)
                    changed = True
        if changed:
            path.write_text(json.dumps(existing, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
            print("merged embedded corrections into existing file:", path)
        else:
            print("corrections file exists:", path)
        data = existing
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(embedded, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print("wrote embedded corrections:", path)
        data = embedded
    print(
        "active corrections:",
        "bbox_corrections=", len(data.get("bbox_corrections", [])),
        "bbox_additions=", len(data.get("bbox_additions", [])),
        "category_corrections=", len(data.get("category_corrections", [])),
    )

ensure_corrections_json(CORRECTIONS_PATH)


## 1. Install & Clone RT-DETRv2

Kaggle Notebook에서 실행하는 기준입니다.  
이미 repo가 있으면 clone은 건너뜁니다.

In [ ]:
import os
from pathlib import Path

WORK_DIR.mkdir(parents=True, exist_ok=True)

if not RTDETRV2_DIR.exists():
    !git clone https://github.com/lyuwenyu/RT-DETR.git {REPO_DIR}
else:
    print("RT-DETR repo already exists:", RTDETRV2_DIR)

os.chdir(RTDETRV2_DIR)
print("cwd:", Path.cwd())

# Colab은 PyTorch/CUDA가 기본 설치되어 있으므로 detection 학습에 필요한 패키지만 추가합니다.
install_packages = [
    "kagglehub",
    "tensorboard",
    "faster-coco-eval",
    "pycocotools",
    "opencv-python-headless",
    "pandas",
    "pillow",
    "tqdm",
    "scikit-learn",
    "matplotlib",
    "PyYAML",
    "scipy",
    "timm",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *install_packages])


## 2. Imports & Reproducibility

In [ ]:
import os
import subprocess
import sys
import json
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(RANDOM_SEED)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("mps available:", torch.backends.mps.is_available() if hasattr(torch.backends, "mps") else False)

def resolve_device(processor="auto"):
    processor = str(processor).lower()
    has_mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    if processor == "cuda" and torch.cuda.is_available():
        return "cuda"
    if processor == "mps" and has_mps:
        return "mps"
    if processor == "cpu":
        return "cpu"
    if processor == "auto":
        if torch.cuda.is_available():
            return "cuda"
        if has_mps:
            return "mps"
    print(f"requested processor={processor!r} is not available; falling back to cpu")
    return "cpu"

TRAIN_DEVICE = resolve_device(PROCESSOR)
print("train/inference device:", TRAIN_DEVICE)

if IN_COLAB and TRAIN_DEVICE != "cuda":
    raise RuntimeError("Colab CUDA notebook인데 CUDA가 잡히지 않았습니다. Runtime > Change runtime type > GPU를 확인하세요.")


## 3. Locate Data

대회마다 폴더명이 다를 수 있어서, 지정 경로가 틀렸을 때 후보를 보여주도록 만들었습니다.

In [ ]:
import zipfile
from tqdm.auto import tqdm

def list_input_tree(root: Path, max_depth=2, max_items=80):
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        if len(rel.parts) <= max_depth:
            rows.append(str(rel) + ("/" if p.is_dir() else ""))
        if len(rows) >= max_items:
            break
    return rows

def looks_like_input_dir(path: Path):
    return (
        path.exists()
        and (path / TRAIN_IMAGE_DIR_NAME).exists()
        and (path / TEST_IMAGE_DIR_NAME).exists()
        and ((path / TRAIN_ANNOTATION_DIR_NAME).exists() or (path / TRAIN_CSV_NAME).exists())
    )

def maybe_load_kaggle_credentials():
    if IN_COLAB:
        try:
            from google.colab import userdata  # type: ignore
            for name in ["KAGGLE_USERNAME", "KAGGLE_KEY"]:
                if not os.environ.get(name):
                    value = userdata.get(name)
                    if value:
                        os.environ[name] = value
        except Exception:
            pass

    kaggle_json_candidates = [
        PROJECT_DIR / "kaggle.json",
        Path("/content/kaggle.json"),
        Path.cwd() / "kaggle.json",
    ]
    kaggle_json = next((p for p in kaggle_json_candidates if p.exists()), None)
    if kaggle_json is not None:
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        target = kaggle_dir / "kaggle.json"
        if kaggle_json.resolve() != target.resolve():
            shutil.copy2(kaggle_json, target)
        target.chmod(0o600)
        print("kaggle.json ready:", target)

def search_roots(extra_roots=None):
    roots = [PROJECT_DIR, Path.cwd()]
    if IN_COLAB:
        roots += [
            Path("/content"),
            Path("/content/detectionproject"),
            Path("/content/drive/MyDrive/detectionproject"),
            Path("/content/drive/MyDrive/AIENGINEERCOURSE/detectionproject"),
        ]
    if extra_roots:
        roots = list(extra_roots) + roots

    seen = set()
    for root in roots:
        root = Path(root).expanduser().resolve()
        if root in seen or not root.exists():
            continue
        seen.add(root)
        yield root

def find_input_dir(roots):
    for root in roots:
        if looks_like_input_dir(root):
            return root
        direct = root / DATA_DIR_NAME
        if looks_like_input_dir(direct):
            return direct
        try:
            for p in root.rglob(DATA_DIR_NAME):
                if looks_like_input_dir(p):
                    return p
        except Exception:
            continue
    return None

def find_zip_path(roots):
    candidates = []
    for root in roots:
        if root.is_file() and root.suffix.lower() == ".zip":
            candidates.append(root)
            continue
        direct = root / DATA_ZIP_NAME
        if direct.exists():
            candidates.append(direct)
        try:
            candidates.extend(root.rglob("*.zip"))
        except Exception:
            pass
    if not candidates:
        return None
    candidates = sorted(set(candidates), key=lambda p: (p.name != DATA_ZIP_NAME, len(str(p)), p.name))
    return candidates[0]

def extract_zip_with_progress(zip_path: Path, dst_dir: Path):
    print("extracting:", zip_path, "->", dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        members = zf.infolist()
        for member in tqdm(members, desc="unzip data", unit="file"):
            zf.extract(member, dst_dir)

def download_competition_with_kagglehub():
    maybe_load_kaggle_credentials()
    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError("kagglehub가 없습니다. Install cell을 먼저 실행하세요.") from exc

    print("downloading competition via kagglehub:", KAGGLE_COMPETITION)
    try:
        downloaded = kagglehub.competition_download(
            KAGGLE_COMPETITION,
            force_download=FORCE_KAGGLE_DOWNLOAD,
        )
    except TypeError:
        downloaded = kagglehub.competition_download(KAGGLE_COMPETITION)

    downloaded_path = Path(downloaded).expanduser().resolve()
    print("Path to competition files:", downloaded_path)
    return downloaded_path

def locate_or_prepare_input_dir():
    global COMPETITION_INPUT_DIR

    roots = list(search_roots())

    if not FORCE_KAGGLE_DOWNLOAD:
        data_dir = find_input_dir(roots)
        if data_dir is not None:
            COMPETITION_INPUT_DIR = data_dir
            return data_dir

        zip_path = find_zip_path(roots)
        if zip_path is not None:
            extract_target = Path("/content") if IN_COLAB else PROJECT_DIR
            extract_zip_with_progress(zip_path, extract_target)
            roots = list(search_roots([extract_target]))
            data_dir = find_input_dir(roots)
            if data_dir is not None:
                COMPETITION_INPUT_DIR = data_dir
                return data_dir

    downloaded_path = download_competition_with_kagglehub()
    roots = list(search_roots([downloaded_path]))

    data_dir = find_input_dir(roots)
    if data_dir is not None:
        COMPETITION_INPUT_DIR = data_dir
        return data_dir

    zip_path = find_zip_path(roots)
    if zip_path is not None:
        extract_target = Path("/content") if IN_COLAB else PROJECT_DIR
        extract_zip_with_progress(zip_path, extract_target)
        roots = list(search_roots([extract_target, downloaded_path]))
        data_dir = find_input_dir(roots)
        if data_dir is not None:
            COMPETITION_INPUT_DIR = data_dir
            return data_dir

    print("❌ input data not found.")
    print("Searched roots:")
    for root in roots:
        print(" -", root)
    if IN_COLAB:
        print("\nKaggle 인증이 필요하면 Colab Secrets에 KAGGLE_USERNAME/KAGGLE_KEY를 넣거나 /content/kaggle.json을 업로드하세요.")
    raise FileNotFoundError("Could not locate sprint_ai_project1_data after kagglehub download")

COMPETITION_INPUT_DIR = locate_or_prepare_input_dir()
print("✅ input dir exists:", COMPETITION_INPUT_DIR)
print("\nPreview:")
for item in list_input_tree(COMPETITION_INPUT_DIR):
    print(" -", item)


In [ ]:
TRAIN_CSV = COMPETITION_INPUT_DIR / TRAIN_CSV_NAME
TRAIN_IMG_DIR = COMPETITION_INPUT_DIR / TRAIN_IMAGE_DIR_NAME
TEST_IMG_DIR = COMPETITION_INPUT_DIR / TEST_IMAGE_DIR_NAME
TRAIN_ANN_DIR = COMPETITION_INPUT_DIR / TRAIN_ANNOTATION_DIR_NAME

# Fallback: 이름이 다르면 csv와 이미지 폴더 후보 자동 탐색
if not TRAIN_CSV.exists() and not TRAIN_ANN_DIR.exists():
    csv_candidates = list(COMPETITION_INPUT_DIR.rglob("*.csv"))
    print("CSV candidates:")
    for p in csv_candidates:
        print(" -", p)
    raise FileNotFoundError(f"TRAIN_CSV not found and TRAIN_ANN_DIR not found: {TRAIN_CSV}, {TRAIN_ANN_DIR}")

if not TRAIN_IMG_DIR.exists():
    print("Image directory candidates:")
    for p in COMPETITION_INPUT_DIR.rglob("*"):
        if p.is_dir() and any(x.suffix.lower() in [".jpg", ".jpeg", ".png"] for x in p.glob("*")):
            print(" -", p)
    raise FileNotFoundError(f"TRAIN_IMG_DIR not found: {TRAIN_IMG_DIR}")

if not TEST_IMG_DIR.exists():
    print("Image directory candidates:")
    for p in COMPETITION_INPUT_DIR.rglob("*"):
        if p.is_dir() and any(x.suffix.lower() in [".jpg", ".jpeg", ".png"] for x in p.glob("*")):
            print(" -", p)
    raise FileNotFoundError(f"TEST_IMG_DIR not found: {TEST_IMG_DIR}")

print("TRAIN_CSV:", TRAIN_CSV)
print("TRAIN_IMG_DIR:", TRAIN_IMG_DIR)
print("TEST_IMG_DIR:", TEST_IMG_DIR)
print("TRAIN_ANN_DIR:", TRAIN_ANN_DIR)

## 4. Load & Validate Train CSV

대회 제출 포맷은 아래 컬럼을 요구합니다.

```text
annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score
```

학습용 `train.csv`에 `score`가 있으면 무시합니다.

In [ ]:
def build_train_csv_from_json_annotations(annotation_dir: Path, out_csv: Path):
    rows = []
    json_paths = sorted(annotation_dir.rglob("*.json"))
    if not json_paths:
        raise FileNotFoundError(f"No annotation json files under {annotation_dir}")

    for json_path in tqdm(json_paths, desc="read annotations"):
        with open(json_path, "r") as f:
            coco = json.load(f)

        images = coco.get("images", [])
        image_by_id = {int(img["id"]): img for img in images}
        default_image = images[0] if images else {}

        for ann in coco.get("annotations", []):
            image_id = int(ann.get("image_id", default_image.get("id")))
            img = image_by_id.get(image_id, default_image)
            bbox_x, bbox_y, bbox_w, bbox_h = ann["bbox"]
            rows.append({
                "annotation_id": int(ann.get("id", len(rows) + 1)),
                "image_id": image_id,
                "file_name": img.get("file_name") or img.get("imgfile"),
                "category_id": int(ann["category_id"]),
                "bbox_x": bbox_x,
                "bbox_y": bbox_y,
                "bbox_w": bbox_w,
                "bbox_h": bbox_h,
            })

    df_built = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    df_built["annotation_id"] = np.arange(1, len(df_built) + 1)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df_built.to_csv(out_csv, index=False)
    return df_built

def apply_annotation_corrections(df_in: pd.DataFrame, corrections_path: Path):
    df_out = df_in.copy()
    if not corrections_path.exists():
        print("corrections file not found; using raw annotations:", corrections_path)
        return df_out

    with open(corrections_path, "r") as f:
        corrections = json.load(f)

    corrected_count = 0
    for item in corrections.get("bbox_corrections", []):
        ann_id = int(item["ann_id"])
        mask = pd.to_numeric(df_out["annotation_id"], errors="coerce").astype("Int64") == ann_id
        if not mask.any():
            print("warning: correction ann_id not found:", ann_id)
            continue
        x, y, w, h = item["corrected"]
        df_out.loc[mask, ["bbox_x", "bbox_y", "bbox_w", "bbox_h"]] = [x, y, w, h]
        corrected_count += int(mask.sum())


    category_corrected_count = 0
    for item in corrections.get("category_corrections", []):
        ann_id = int(item["ann_id"])
        mask = pd.to_numeric(df_out["annotation_id"], errors="coerce").astype("Int64") == ann_id
        if not mask.any():
            print("warning: category correction ann_id not found:", ann_id)
            continue
        expected = item.get("original_category_id")
        if expected is not None:
            current = pd.to_numeric(df_out.loc[mask, "category_id"], errors="coerce").astype("Int64")
            mismatched = current.notna() & current.ne(int(expected)) & current.ne(int(item["corrected_category_id"]))
            if bool(mismatched.any()):
                print("warning: category correction original mismatch:", ann_id, "current=", current.tolist())
        df_out.loc[mask, "category_id"] = int(item["corrected_category_id"])
        category_corrected_count += int(mask.sum())

    existing_ann_ids = set(pd.to_numeric(df_out["annotation_id"], errors="coerce").dropna().astype(int))
    additions = []
    for item in corrections.get("bbox_additions", []):
        ann_id = int(item["ann_id"])
        if ann_id in existing_ann_ids:
            print("warning: addition ann_id already exists, skipped:", ann_id)
            continue
        x, y, w, h = item["bbox"]
        additions.append({
            "annotation_id": ann_id,
            "image_id": int(item["image_id"]),
            "file_name": item.get("file_name"),
            "category_id": int(item["category_id"]),
            "bbox_x": x,
            "bbox_y": y,
            "bbox_w": w,
            "bbox_h": h,
        })

    if additions:
        df_out = pd.concat([df_out, pd.DataFrame(additions)], ignore_index=True)

    print(f"applied corrections: bbox_corrections={corrected_count}, bbox_additions={len(additions)}, category_corrections={category_corrected_count}")
    return df_out

if TRAIN_CSV.exists():
    df_raw = pd.read_csv(TRAIN_CSV)
else:
    TRAIN_CSV = WORK_DIR / TRAIN_CSV_NAME
    df_raw = build_train_csv_from_json_annotations(TRAIN_ANN_DIR, TRAIN_CSV)
    print("generated train csv:", TRAIN_CSV)

df_raw = apply_annotation_corrections(df_raw, CORRECTIONS_PATH)

print("shape:", df_raw.shape)
display(df_raw.head())
print(df_raw.dtypes)

In [ ]:
REQUIRED_COLUMNS = ["image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h"]

missing = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df_raw.copy()

if "file_name" in df.columns:
    IMAGE_ID_TO_FILE = (
        df[["image_id", "file_name"]]
        .dropna()
        .drop_duplicates("image_id")
        .assign(image_id=lambda x: pd.to_numeric(x["image_id"], errors="raise").astype(int))
        .set_index("image_id")["file_name"]
        .to_dict()
    )
else:
    IMAGE_ID_TO_FILE = {}

# annotation_id가 없으면 생성
if "annotation_id" not in df.columns:
    df["annotation_id"] = np.arange(1, len(df) + 1)

# score는 학습에서 제외
df = df[["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h"]].copy()

# 숫자 변환
for c in ["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h"]:
    df[c] = pd.to_numeric(df[c], errors="raise")

print("num annotations:", len(df))
print("num images:", df["image_id"].nunique())
print("num categories:", df["category_id"].nunique())
print("image filename mappings:", len(IMAGE_ID_TO_FILE))
display(df.describe())

## 5. Image Path Helper

파일명이 `1.jpg`, `000001.jpg`, `1.png`처럼 다를 수 있어서 안전하게 찾습니다.

In [ ]:
def find_image_path(img_dir: Path, image_id):
    image_id_int = int(image_id)
    mapped_name = IMAGE_ID_TO_FILE.get(image_id_int)
    if mapped_name:
        p = img_dir / mapped_name
        if p.exists():
            return p
    candidates = [
        f"{image_id_int}.jpg",
        f"{image_id_int}.jpeg",
        f"{image_id_int}.png",
        f"{image_id_int:06d}.jpg",
        f"{image_id_int:06d}.jpeg",
        f"{image_id_int:06d}.png",
    ]

    for name in candidates:
        p = img_dir / name
        if p.exists():
            return p

    # fallback: stem 숫자가 image_id와 같은 파일
    for p in img_dir.glob("*"):
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"] and p.stem.isdigit():
            if int(p.stem) == image_id_int:
                return p

    raise FileNotFoundError(f"image_id={image_id} file not found under {img_dir}")

# quick check
sample_ids = df["image_id"].drop_duplicates().head(5).tolist()
for sid in sample_ids:
    print(sid, "->", find_image_path(TRAIN_IMG_DIR, sid).name)

## 6. Category Mapping

RT-DETR 학습에는 category id를 `0..N-1`로 연속화해서 넣고, 제출 때 원래 `category_id`로 되돌립니다.

In [ ]:
original_categories = sorted(df["category_id"].unique().tolist())
cat2idx = {int(cat): int(i) for i, cat in enumerate(original_categories)}
idx2cat = {int(i): int(cat) for cat, i in cat2idx.items()}

df["category_idx"] = df["category_id"].map(cat2idx).astype(int)

NUM_CLASSES = len(original_categories)

DATA_WORK_DIR.mkdir(parents=True, exist_ok=True)
with open(DATA_WORK_DIR / "idx2cat.json", "w") as f:
    json.dump(idx2cat, f, indent=2)

print("NUM_CLASSES:", NUM_CLASSES)
print("mapping sample:", list(cat2idx.items())[:10])

## 7. Train / Validation Split

이미지 단위로 split합니다. 같은 이미지의 여러 bbox가 train/val에 섞이지 않게 막습니다.

파일명 기준으로 `70/75/90`처럼 촬영각만 다른 near-duplicate는 같은 그룹으로 묶어 train/val 누수를 막습니다.

In [ ]:
def image_group_key(image_id):
    file_name = IMAGE_ID_TO_FILE.get(int(image_id))
    if not file_name:
        return str(int(image_id))

    stem = Path(file_name).stem
    parts = stem.split("_")
    # K-..._0_2_0_2_70_000_200 -> remove only the camera-angle token.
    if len(parts) >= 8 and parts[0].startswith("K-"):
        return "_".join(parts[:5] + parts[6:])
    return stem

image_ids = sorted(df["image_id"].unique().tolist())
image_id_to_group = {int(image_id): image_group_key(image_id) for image_id in image_ids}
unique_groups = np.array(sorted(set(image_id_to_group.values())))

rng = np.random.RandomState(RANDOM_SEED)
rng.shuffle(unique_groups)
n_val_groups = max(1, int(np.ceil(len(unique_groups) * VAL_SIZE)))
val_groups = set(unique_groups[:n_val_groups].tolist())

train_ids = {int(image_id) for image_id in image_ids if image_id_to_group[int(image_id)] not in val_groups}
val_ids = {int(image_id) for image_id in image_ids if image_id_to_group[int(image_id)] in val_groups}

train_groups = {image_id_to_group[int(image_id)] for image_id in train_ids}
assert train_groups.isdisjoint(val_groups), "near-duplicate group leaked across train/val"

print("train images:", len(train_ids))
print("val images:", len(val_ids))
print("train groups:", len(train_groups))
print("val groups:", len(val_groups))
print("train annotations:", df[df["image_id"].isin(train_ids)].shape[0])
print("val annotations:", df[df["image_id"].isin(val_ids)].shape[0])

## 8. Convert to COCO Detection Format

RT-DETRv2는 COCO style detection annotation을 사용합니다.

COCO bbox 형식:

```text
[x, y, width, height]
```

In [ ]:
IMG_TRAIN_OUT = DATA_WORK_DIR / "images" / "train"
IMG_VAL_OUT = DATA_WORK_DIR / "images" / "val"
ANN_DIR = DATA_WORK_DIR / "annotations"

for p in [IMG_TRAIN_OUT, IMG_VAL_OUT, ANN_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def safe_copy(src: Path, dst: Path):
    if not dst.exists():
        shutil.copy2(src, dst)

def clamp_bbox_xywh(x, y, bw, bh, img_w, img_h):
    x = float(max(0, min(x, img_w - 1)))
    y = float(max(0, min(y, img_h - 1)))
    bw = float(max(1, min(bw, img_w - x)))
    bh = float(max(1, min(bh, img_h - y)))
    return x, y, bw, bh

def make_coco_json(split_ids, out_img_dir: Path, out_json_path: Path):
    images = []
    annotations = []
    ann_id = 1

    split_df = df[df["image_id"].isin(split_ids)].copy()

    for image_id in tqdm(sorted(split_ids), desc=f"COCO {out_json_path.name}"):
        src_path = find_image_path(TRAIN_IMG_DIR, image_id)
        dst_path = out_img_dir / src_path.name
        safe_copy(src_path, dst_path)

        with Image.open(src_path) as im:
            img_w, img_h = im.size

        images.append({
            "id": int(image_id),
            "file_name": src_path.name,
            "width": int(img_w),
            "height": int(img_h),
        })

        rows = split_df[split_df["image_id"] == image_id]
        for _, row in rows.iterrows():
            x, y, bw, bh = clamp_bbox_xywh(
                row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"], img_w, img_h
            )

            annotations.append({
                "id": int(ann_id),
                "image_id": int(image_id),
                "category_id": int(row["category_idx"]),
                "bbox": [x, y, bw, bh],
                "area": float(bw * bh),
                "iscrowd": 0,
            })
            ann_id += 1

    categories = [
        {"id": int(i), "name": str(idx2cat[i])}
        for i in range(NUM_CLASSES)
    ]

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": categories,
    }

    with open(out_json_path, "w") as f:
        json.dump(coco, f)

    print("saved:", out_json_path)
    print("images:", len(images), "annotations:", len(annotations), "categories:", len(categories))

make_coco_json(train_ids, IMG_TRAIN_OUT, ANN_DIR / "train.json")
make_coco_json(val_ids, IMG_VAL_OUT, ANN_DIR / "val.json")

## 9. Quick Visual Check

라벨 box가 이미지 위에 제대로 얹히는지 확인합니다.  
고 IoU 대회에서는 이 단계가 은근히 왕관 쓴 문지기입니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def show_ground_truth(image_id):
    img_path = find_image_path(TRAIN_IMG_DIR, image_id)
    im = Image.open(img_path).convert("RGB")
    rows = df[df["image_id"] == image_id]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(im)
    ax.set_title(f"GT image_id={image_id}, boxes={len(rows)}")
    ax.axis("off")

    for _, r in rows.iterrows():
        rect = patches.Rectangle(
            (r["bbox_x"], r["bbox_y"]),
            r["bbox_w"],
            r["bbox_h"],
            fill=False,
            linewidth=2,
        )
        ax.add_patch(rect)
        ax.text(
            r["bbox_x"],
            max(0, r["bbox_y"] - 5),
            str(int(r["category_id"])),
            fontsize=10,
            bbox=dict(facecolor="white", alpha=0.7),
        )
    plt.show()

show_ground_truth(next(iter(train_ids)))

## 10. Create RT-DETRv2-X Custom Config

공식 RT-DETRv2 PyTorch 구조를 사용합니다.

- `MODEL_SIZE="x"`: RT-DETRv2-X, R101 backbone
- `MODEL_SIZE="l"`: RT-DETRv2-L, R50 backbone fallback
- `num_classes`: 대회 category 개수
- `remap_mscoco_category=False`: COCO 80-class remap 끄기

In [ ]:
os.chdir(RTDETRV2_DIR)
print("cwd:", Path.cwd())

CONFIG_DIR = RTDETRV2_DIR / "configs"
CUSTOM_DATASET_YML = CONFIG_DIR / "dataset" / "pill_detection.yml"

dataset_yml = f'''
task: detection

evaluator:
  type: CocoEvaluator
  iou_types: ['bbox']

num_classes: {NUM_CLASSES}
remap_mscoco_category: False

train_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: {str(IMG_TRAIN_OUT)}
    ann_file: {str(ANN_DIR / "train.json")}
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: True
  num_workers: {NUM_WORKERS}
  drop_last: True
  collate_fn:
    type: BatchImageCollateFunction

val_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: {str(IMG_VAL_OUT)}
    ann_file: {str(ANN_DIR / "val.json")}
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: False
  num_workers: {NUM_WORKERS}
  drop_last: False
  collate_fn:
    type: BatchImageCollateFunction
'''

CUSTOM_DATASET_YML.write_text(dataset_yml)
print(CUSTOM_DATASET_YML.read_text())

In [ ]:
CUSTOM_MODEL_YML = CONFIG_DIR / "rtdetrv2" / f"rtdetrv2_{MODEL_SIZE}_pill.yml"

# Official RT-DETRv2-X config is based on r101vd_6x_coco.
# RT-DETRv2-L config is based on r50vd_6x_coco.
if MODEL_SIZE.lower() == "x":
    model_body = f'''
__include__: [
  '../dataset/pill_detection.yml',
  '../runtime.yml',
  './include/dataloader.yml',
  './include/optimizer.yml',
  './include/rtdetrv2_r50vd.yml',
]

output_dir: ./output/rtdetrv2_x_pill

PResNet:
  depth: 101

HybridEncoder:
  hidden_dim: 384
  dim_feedforward: 2048

RTDETRTransformerv2:
  feat_channels: [384, 384, 384]

train_dataloader:
  total_batch_size: {TOTAL_BATCH_SIZE}
  num_workers: {NUM_WORKERS}
  dataset:
    transforms:
      ops:
        - {{type: RandomPhotometricDistort, p: 0.5}}
        - {{type: RandomZoomOut, fill: 0}}
        - {{type: RandomIoUCrop, p: 0.8}}
        - {{type: SanitizeBoundingBoxes, min_size: 1}}
        - {{type: RandomHorizontalFlip}}
        - {{type: Resize, size: [{IMAGE_SIZE}, {IMAGE_SIZE}]}}
        - {{type: SanitizeBoundingBoxes, min_size: 1}}
        - {{type: ConvertPILImage, dtype: 'float32', scale: True}}
        - {{type: ConvertBoxes, fmt: 'cxcywh', normalize: True}}

val_dataloader:
  total_batch_size: {max(1, TOTAL_BATCH_SIZE)}
  num_workers: {NUM_WORKERS}
  dataset:
    transforms:
      ops:
        - {{type: Resize, size: [{IMAGE_SIZE}, {IMAGE_SIZE}]}}
        - {{type: ConvertPILImage, dtype: 'float32', scale: True}}

optimizer:
  type: AdamW
  params:
    -
      params: '^(?=.*backbone)(?!.*norm|bn).*$'
      lr: 0.000001
    -
      params: '^(?=.*(?:encoder|decoder))(?=.*(?:norm|bn)).*$'
      weight_decay: 0.
  lr: 0.0001
  betas: [0.9, 0.999]
  weight_decay: 0.0001

epoches: {EPOCHS}
use_amp: {str(USE_AMP)}
'''
elif MODEL_SIZE.lower() == "l":
    model_body = f'''
__include__: [
  '../dataset/pill_detection.yml',
  '../runtime.yml',
  './include/dataloader.yml',
  './include/optimizer.yml',
  './include/rtdetrv2_r50vd.yml',
]

output_dir: ./output/rtdetrv2_l_pill

train_dataloader:
  total_batch_size: {TOTAL_BATCH_SIZE}
  num_workers: {NUM_WORKERS}
  dataset:
    transforms:
      ops:
        - {{type: RandomPhotometricDistort, p: 0.5}}
        - {{type: RandomZoomOut, fill: 0}}
        - {{type: RandomIoUCrop, p: 0.8}}
        - {{type: SanitizeBoundingBoxes, min_size: 1}}
        - {{type: RandomHorizontalFlip}}
        - {{type: Resize, size: [{IMAGE_SIZE}, {IMAGE_SIZE}]}}
        - {{type: SanitizeBoundingBoxes, min_size: 1}}
        - {{type: ConvertPILImage, dtype: 'float32', scale: True}}
        - {{type: ConvertBoxes, fmt: 'cxcywh', normalize: True}}

val_dataloader:
  total_batch_size: {max(1, TOTAL_BATCH_SIZE)}
  num_workers: {NUM_WORKERS}
  dataset:
    transforms:
      ops:
        - {{type: Resize, size: [{IMAGE_SIZE}, {IMAGE_SIZE}]}}
        - {{type: ConvertPILImage, dtype: 'float32', scale: True}}

epoches: {EPOCHS}
use_amp: {str(USE_AMP)}
'''
else:
    raise ValueError("MODEL_SIZE must be 'x' or 'l'")

CUSTOM_MODEL_YML.write_text(model_body)
print("custom config:", CUSTOM_MODEL_YML)
print(CUSTOM_MODEL_YML.read_text())

## 11. Download Pretrained Checkpoint

공식 checkpoint를 fine-tuning 시작점으로 사용합니다.

- X: `rtdetrv2_r101vd_6x_coco_from_paddle.pth`
- L: `rtdetrv2_r50vd_6x_coco_from_paddle.pth`

In [ ]:
PRETRAIN_DIR = WORK_DIR / "pretrained"
PRETRAIN_DIR.mkdir(parents=True, exist_ok=True)

if MODEL_SIZE.lower() == "x":
    PRETRAINED_CKPT = PRETRAIN_DIR / "rtdetrv2_r101vd_6x_coco_from_paddle.pth"
    PRETRAINED_URL = "https://github.com/lyuwenyu/storage/releases/download/v0.1/rtdetrv2_r101vd_6x_coco_from_paddle.pth"
else:
    PRETRAINED_CKPT = PRETRAIN_DIR / "rtdetrv2_r50vd_6x_coco_from_paddle.pth"
    PRETRAINED_URL = "https://github.com/lyuwenyu/storage/releases/download/v0.1/rtdetrv2_r50vd_6x_coco_from_paddle.pth"

def download_with_tqdm(url, dst: Path):
    import urllib.request
    from tqdm.auto import tqdm

    dst.parent.mkdir(parents=True, exist_ok=True)
    with tqdm(unit="B", unit_scale=True, unit_divisor=1024, desc=dst.name) as pbar:
        def reporthook(block_num, block_size, total_size):
            if total_size > 0 and pbar.total != total_size:
                pbar.total = total_size
            downloaded = block_num * block_size
            pbar.update(max(0, downloaded - pbar.n))

        urllib.request.urlretrieve(url, dst, reporthook=reporthook)
        pbar.n = pbar.total or pbar.n
        pbar.refresh()

if not PRETRAINED_CKPT.exists():
    print("downloading:", PRETRAINED_URL)
    download_with_tqdm(PRETRAINED_URL, PRETRAINED_CKPT)
else:
    print("pretrained checkpoint exists:", PRETRAINED_CKPT)

print(PRETRAINED_CKPT, PRETRAINED_CKPT.exists(), f"{PRETRAINED_CKPT.stat().st_size / 1024**2:.1f} MB" if PRETRAINED_CKPT.exists() else "")

## 12. Train RT-DETRv2

OOM이 나면 아래 순서로 낮추세요.

```text
TOTAL_BATCH_SIZE=4 → 2 → 1
MODEL_SIZE="x" → "l"
IMAGE_SIZE=640 유지
```

In [ ]:
import os
import re
import subprocess
import sys
from tqdm.auto import tqdm

for var in ["RANK", "LOCAL_RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]:
    os.environ.pop(var, None)

os.chdir(RTDETRV2_DIR)
print("cwd:", Path.cwd())

train_cmd = [
    sys.executable,
    "tools/train.py",
    "-c", str(CUSTOM_MODEL_YML),
    "-t", str(PRETRAINED_CKPT),
    "-d", TRAIN_DEVICE,
    f"--seed={RANDOM_SEED}",
    "-u", "print_freq=10",
]
if USE_AMP:
    train_cmd.append("--use-amp")

print("run:", " ".join(train_cmd))

epoch_re = re.compile(r"Epoch:\s*\[(\d+)\]\s*\[\s*(\d+)/(\d+)\]")
epoch_done_re = re.compile(r"Epoch:\s*\[(\d+)\]\s*Total time")
important_prefixes = (
    "Start training",
    "Not init distributed mode",
    "Initialized distributed mode",
    "Averaged stats:",
    "Test:",
    "Accumulating evaluation results",
    "DONE",
)
important_contains = (
    "IoU metric:",
    "Average Precision",
    "Average Recall",
    "coco_eval_bbox",
    "best_stat",
    "warning",
    "Warning",
    "ERROR",
    "Error",
    "Traceback",
)

cfg_omitted = False

def should_print_train_line(line: str):
    stripped = line.strip()
    if not stripped:
        return False
    if stripped.startswith("cfg:"):
        return False
    if stripped.startswith("Epoch:"):
        return True
    if stripped.startswith(important_prefixes):
        return True
    return any(token in stripped for token in important_contains)

epoch_bar = tqdm(total=EPOCHS, desc="epochs", unit="epoch")
iter_bar = None
iter_epoch = None
try:
    process = subprocess.Popen(
        train_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=os.environ.copy(),
    )

    assert process.stdout is not None
    for line in process.stdout:
        if line.startswith("cfg:") and not cfg_omitted:
            print("cfg: <omitted; custom YAML files were printed in the previous config cells>")
            cfg_omitted = True
        elif should_print_train_line(line):
            print(line, end="")

        match = epoch_re.search(line)
        if match:
            epoch = int(match.group(1))
            step = int(match.group(2))
            total = int(match.group(3))
            if iter_bar is None or iter_epoch != epoch:
                if iter_bar is not None:
                    iter_bar.close()
                iter_epoch = epoch
                iter_bar = tqdm(total=total, desc=f"epoch {epoch + 1}/{EPOCHS}", unit="it", leave=False)
            iter_bar.n = min(step + 1, total)
            iter_bar.refresh()

        done = epoch_done_re.search(line)
        if done:
            epoch = int(done.group(1))
            epoch_bar.n = max(epoch_bar.n, min(epoch + 1, EPOCHS))
            epoch_bar.refresh()
            if iter_bar is not None:
                iter_bar.n = iter_bar.total or iter_bar.n
                iter_bar.refresh()
                iter_bar.close()
                iter_bar = None
                iter_epoch = None

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, train_cmd)
finally:
    if iter_bar is not None:
        iter_bar.close()
    epoch_bar.close()

## 13. Find Best Checkpoint

보통 `best.pth`, `last.pth`, 또는 epoch별 checkpoint가 생성됩니다.  
없으면 output folder를 먼저 확인하세요.

In [ ]:
OUTPUT_DIR = RTDETRV2_DIR / "output" / f"rtdetrv2_{MODEL_SIZE.lower()}_pill"

print("OUTPUT_DIR:", OUTPUT_DIR)
if OUTPUT_DIR.exists():
    for p in sorted(OUTPUT_DIR.glob("*")):
        print(p.name)
else:
    print("output dir does not exist yet")

ckpt_candidates = []
if OUTPUT_DIR.exists():
    preferred = ["best.pth", "ema_best.pth", "last.pth", "checkpoint.pth"]
    for name in preferred:
        p = OUTPUT_DIR / name
        if p.exists():
            ckpt_candidates.append(p)
    ckpt_candidates += sorted(OUTPUT_DIR.glob("*.pth"))

ckpt_candidates = list(dict.fromkeys(ckpt_candidates))

if not ckpt_candidates:
    raise FileNotFoundError("No checkpoint found. Check training output folder.")

BEST_CKPT = ckpt_candidates[0]
print("BEST_CKPT:", BEST_CKPT)

## 14. Official Test-Only Validation

공식 evaluator로 validation 성능을 확인합니다.

In [ ]:
import os
import subprocess
import sys
from tqdm.auto import tqdm

for var in ["RANK", "LOCAL_RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]:
    os.environ.pop(var, None)

os.chdir(RTDETRV2_DIR)
print("cwd:", Path.cwd())

test_cmd = [
    sys.executable,
    "tools/train.py",
    "-c", str(CUSTOM_MODEL_YML),
    "-r", str(BEST_CKPT),
    "-d", TRAIN_DEVICE,
    "--test-only",
]

print("run:", " ".join(test_cmd))
with tqdm(total=1, desc="official validation", unit="run") as pbar:
    process = subprocess.Popen(
        test_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=os.environ.copy(),
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    pbar.update(1)
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, test_cmd)

## 15. Load Model for Custom Inference

아래 모델 로더는 validation 예측과 submission 생성을 같이 사용합니다.

In [ ]:
os.chdir(RTDETRV2_DIR)
print("cwd:", Path.cwd())

import sys
import torch
import torch.nn as nn
import torchvision.transforms as T

sys.path.append(str(RTDETRV2_DIR))

from src.core import YAMLConfig

DEVICE = TRAIN_DEVICE
CONFIG_PATH = str(CUSTOM_MODEL_YML)
CKPT_PATH = str(BEST_CKPT)

cfg = YAMLConfig(CONFIG_PATH, resume=CKPT_PATH)

checkpoint = torch.load(CKPT_PATH, map_location="cpu")
if "ema" in checkpoint:
    state = checkpoint["ema"]["module"]
elif "model" in checkpoint:
    state = checkpoint["model"]
else:
    state = checkpoint

cfg.model.load_state_dict(state, strict=False)

class RTDETRv2InferenceModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.model = cfg.model.deploy()
        self.postprocessor = cfg.postprocessor.deploy()

    def forward(self, images, orig_target_sizes):
        outputs = self.model(images)
        outputs = self.postprocessor(outputs, orig_target_sizes)
        return outputs

model = RTDETRv2InferenceModel(cfg).to(DEVICE)
model.eval()

infer_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
])

with open(DATA_WORK_DIR / "idx2cat.json") as f:
    idx2cat = {int(k): int(v) for k, v in json.load(f).items()}

print("loaded model on", DEVICE)

## 16. Inference Helpers

In [ ]:
def get_numeric_image_id_from_path(path: Path):
    stem = path.stem
    if stem.isdigit():
        return int(stem)
    # fallback: 파일명 안 숫자 추출
    digits = "".join(ch for ch in stem if ch.isdigit())
    if digits:
        return int(digits)
    raise ValueError(f"Cannot parse numeric image_id from filename: {path.name}")

def predict_image(img_path: Path, score_thr=SCORE_THR, max_det=MAX_DET_PER_IMAGE):
    im_pil = Image.open(img_path).convert("RGB")
    img_w, img_h = im_pil.size

    orig_size = torch.tensor([[img_w, img_h]], device=DEVICE)
    im_data = infer_transform(im_pil)[None].to(DEVICE)

    with torch.no_grad():
        labels, boxes, scores = model(im_data, orig_size)

    labels = labels[0].detach().cpu()
    boxes = boxes[0].detach().cpu()
    scores = scores[0].detach().cpu()

    keep = scores > score_thr
    labels = labels[keep]
    boxes = boxes[keep]
    scores = scores[keep]

    order = torch.argsort(scores, descending=True)
    if max_det is not None:
        order = order[:max_det]

    labels = labels[order]
    boxes = boxes[order]
    scores = scores[order]

    preds = []
    for lab, box, score in zip(labels, boxes, scores):
        x1, y1, x2, y2 = box.tolist()

        x1 = max(0, min(float(x1), img_w - 1))
        y1 = max(0, min(float(y1), img_h - 1))
        x2 = max(0, min(float(x2), img_w))
        y2 = max(0, min(float(y2), img_h))

        bw = max(1.0, x2 - x1)
        bh = max(1.0, y2 - y1)

        pred_idx = int(lab.item())
        category_id = int(idx2cat.get(pred_idx, pred_idx))

        preds.append({
            "category_id": category_id,
            "category_idx": pred_idx,
            "bbox_x": x1,
            "bbox_y": y1,
            "bbox_w": bw,
            "bbox_h": bh,
            "score": float(score.item()),
        })

    return preds

## 17. Custom Validation mAP@[0.75:0.95]

대회 metric에 맞춰 IoU thresholds를 `[0.75, 0.80, 0.85, 0.90, 0.95]`로 둔 COCOeval을 돌립니다.

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

VAL_JSON = ANN_DIR / "val.json"

def run_val_predictions():
    val_coco = COCO(str(VAL_JSON))
    image_id_to_file = {img["id"]: img["file_name"] for img in val_coco.dataset["images"]}

    coco_results = []
    for image_id, file_name in tqdm(image_id_to_file.items(), desc="val inference"):
        img_path = IMG_VAL_OUT / file_name
        preds = predict_image(img_path, score_thr=SCORE_THR, max_det=MAX_DET_PER_IMAGE)

        for p in preds:
            # COCOeval은 학습용 contiguous category_idx를 기대함
            coco_results.append({
                "image_id": int(image_id),
                "category_id": int(p["category_idx"]),
                "bbox": [
                    float(p["bbox_x"]),
                    float(p["bbox_y"]),
                    float(p["bbox_w"]),
                    float(p["bbox_h"]),
                ],
                "score": float(p["score"]),
            })

    return coco_results

val_results = run_val_predictions()
print("num val predictions:", len(val_results))

VAL_RESULT_JSON = DATA_WORK_DIR / "val_predictions.json"
with open(VAL_RESULT_JSON, "w") as f:
    json.dump(val_results, f)

coco_gt = COCO(str(VAL_JSON))
coco_dt = coco_gt.loadRes(str(VAL_RESULT_JSON))

coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.params.iouThrs = np.arange(0.75, 0.96, 0.05)
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

# 첫 번째 stats가 지정 iouThrs 전체 평균 AP입니다.
print("Custom mAP@[0.75:0.95]:", coco_eval.stats[0])

## 18. Visualize Validation Predictions

GT와 Prediction을 나란히 확인합니다.  
bbox가 살짝이라도 헐거우면 `mAP@[0.75:0.95]`에서 점수가 녹을 수 있습니다.

In [ ]:
def show_predictions(image_id, score_thr=0.05):
    src_path = find_image_path(TRAIN_IMG_DIR, image_id)
    im = Image.open(src_path).convert("RGB")

    gt_rows = df[df["image_id"] == image_id]
    preds = predict_image(src_path, score_thr=score_thr, max_det=MAX_DET_PER_IMAGE)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    axes[0].imshow(im)
    axes[0].set_title(f"GT image_id={image_id}")
    axes[0].axis("off")

    for _, r in gt_rows.iterrows():
        rect = patches.Rectangle(
            (r["bbox_x"], r["bbox_y"]),
            r["bbox_w"],
            r["bbox_h"],
            fill=False,
            linewidth=2,
        )
        axes[0].add_patch(rect)
        axes[0].text(r["bbox_x"], max(0, r["bbox_y"] - 5), str(int(r["category_id"])),
                     fontsize=10, bbox=dict(facecolor="white", alpha=0.7))

    axes[1].imshow(im)
    axes[1].set_title(f"Pred score>{score_thr}")
    axes[1].axis("off")

    for p in preds:
        rect = patches.Rectangle(
            (p["bbox_x"], p["bbox_y"]),
            p["bbox_w"],
            p["bbox_h"],
            fill=False,
            linewidth=2,
        )
        axes[1].add_patch(rect)
        axes[1].text(
            p["bbox_x"],
            max(0, p["bbox_y"] - 5),
            f'{p["category_id"]}:{p["score"]:.2f}',
            fontsize=10,
            bbox=dict(facecolor="white", alpha=0.7),
        )

    plt.show()

sample_val_ids = list(val_ids)[:3]
for sid in sample_val_ids:
    show_predictions(sid, score_thr=0.05)

## 19. Test Inference → submission.csv

제출 포맷:

```text
annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score
```

`image_id`는 파일명의 숫자입니다.

In [ ]:
test_paths = sorted(
    list(TEST_IMG_DIR.glob("*.jpg")) +
    list(TEST_IMG_DIR.glob("*.jpeg")) +
    list(TEST_IMG_DIR.glob("*.png")),
    key=get_numeric_image_id_from_path,
)

print("num test images:", len(test_paths))
print("sample:", [p.name for p in test_paths[:5]])

In [ ]:
submission_rows = []
annotation_id = 1

for img_path in tqdm(test_paths, desc="test inference"):
    image_id = get_numeric_image_id_from_path(img_path)
    preds = predict_image(img_path, score_thr=SCORE_THR, max_det=MAX_DET_PER_IMAGE)

    for p in preds:
        submission_rows.append({
            "annotation_id": int(annotation_id),
            "image_id": int(image_id),
            "category_id": int(p["category_id"]),
            "bbox_x": round(float(p["bbox_x"]), 2),
            "bbox_y": round(float(p["bbox_y"]), 2),
            "bbox_w": round(float(p["bbox_w"]), 2),
            "bbox_h": round(float(p["bbox_h"]), 2),
            "score": round(float(p["score"]), 6),
        })
        annotation_id += 1

submission = pd.DataFrame(
    submission_rows,
    columns=[
        "annotation_id",
        "image_id",
        "category_id",
        "bbox_x",
        "bbox_y",
        "bbox_w",
        "bbox_h",
        "score",
    ],
)

SUBMISSION_PATH = WORK_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

print("saved:", SUBMISSION_PATH)
print("shape:", submission.shape)
display(submission.head(20))

## 20. Submission Sanity Checks

In [ ]:
assert list(submission.columns) == [
    "annotation_id",
    "image_id",
    "category_id",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "score",
]

assert submission["annotation_id"].is_unique
assert submission["bbox_w"].min() > 0
assert submission["bbox_h"].min() > 0
assert submission["score"].between(0, 1).all()

print("✅ submission format looks good")
print("objects per image:")
display(submission.groupby("image_id").size().describe())

## 21. Next Experiments

추천 실험 순서:

```text
Exp 1. RT-DETRv2-X 640, 50 epochs
Exp 2. RT-DETRv2-X 800, 30 epochs
Exp 3. RT-DETRv2-L 800, OOM 대비
Exp 4. RT-DETRv2-X + Swin-T crop classifier
```

대회 metric이 `mAP@[0.75:0.95]`라서 우선 봐야 할 것은:

1. `Recall`: 알약을 놓치지 않는가?
2. `bbox tightness`: box가 약을 꽉 잡는가?
3. `category confusion`: box는 맞는데 class가 틀리는가?
4. `false positive`: 최대 4개 제한에서 쓸데없는 box가 끼는가?

category confusion이 크면 detector를 무한히 키우기보다, `crop → Swin-T classifier`를 붙이는 쪽이 더 깔끔합니다.